In [32]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [33]:
import torch

In [34]:
import plotly.express as px
import pandas as pd
### plot point cloud on cylindrical coordinates using plotly
def plot_point_cloud(results, xy_range):
    df = pd.DataFrame(results)
    fig = px.scatter_3d(df, x='x', y='y', z='z', color='E', size='size', size_max=5)
    fig.update_traces(marker=dict(line=dict(width=0)))
    fig.update_layout(
        width=1000,
        height=800,
        scene=dict(
            xaxis=dict(range=[-xy_range, xy_range]),
            yaxis=dict(range=[-xy_range, xy_range]),
        )
    )
    fig.show()

## Dataset

In [35]:
from dataset import SimplePflowDataset

config = 'simple_pflow.yaml'
dataset = SimplePflowDataset(config, batch_size=10)

In [36]:
for batch in dataset:
    X, y = batch
    print(X.keys())
    print(y.keys())
    break

dict_keys(['cell_x', 'cell_y', 'cell_z', 'cell_e'])
dict_keys(['particle_E', 'particle_x', 'particle_y', 'incidence_matrix'])


In [37]:
X['cell_x'].shape

torch.Size([10, 1011])

In [40]:
from dataset import transform

ev = 0

is_nan = torch.isnan(X['cell_x'][ev])
plot_dict = {k: transform(v[ev][~is_nan], k, dataset.xform_cfg, inverse=True) for k, v in X.items()}
plot_dict['x'] = plot_dict.pop('cell_x')
plot_dict['y'] = plot_dict.pop('cell_y')
plot_dict['z'] = plot_dict.pop('cell_z')
plot_dict['E'] = plot_dict.pop('cell_e')
plot_dict['size'] = (plot_dict['E'] + 1).log()

plot_point_cloud(plot_dict, xy_range=dataset.calo_cfg['width'])